# Session 4. Context and short-term memory

**The session-1 message list becomes a budgeted, persisted resource.**

- remember between invokes: checkpointer and `thread_id`
- survive a restart: `SqliteSaver`, one changed argument
- stop growing forever: window, summary, hard cap

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## A dialogue that outgrows the window

**A support triage agent. No tools, so the token curves stay clean.**

- the user states the key fact once: an order number, in turn 1
- standing rules ride along as a system message on every call
- every kept message is resent every call: the session-1 list, still
- tools return in session 5; today they would only add noise

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph

RULES = (
    "You are a support triage assistant. Keep replies to two sentences, "
    "ask for a missing fact once, and never invent order status."
)
model = chat_model("cheap")  # one model object for the whole session


def call_support(state: MessagesState) -> dict:
    prompt = [SystemMessage(RULES)] + state["messages"]  # rules never live in state
    return {"messages": [model.invoke(prompt)]}


support_builder = StateGraph(MessagesState)
support_builder.add_node("support", call_support)
support_builder.add_edge(START, "support")
support_builder.add_edge("support", END)
amnesiac = support_builder.compile()  # no checkpointer yet, on purpose

first = amnesiac.invoke(
    {"messages": [HumanMessage("Hi. My order ORD-7431 arrived with a cracked screen.")]},
    config={"recursion_limit": 5},
)
second = amnesiac.invoke(
    {"messages": [HumanMessage("What was my order number again?")]},
    config={"recursion_limit": 5},
)
print("first run: ", len(first["messages"]), "messages")
print("second run:", len(second["messages"]), "messages")  # the order number is gone

**Two invokes, two strangers. Session 1 all over again.**

- the graph owns state during a run, not between runs
- `invoke` returns the final state, and then it is garbage
- the fix is not a bigger prompt; it is persistence

## Checkpointer mechanics

**`compile(checkpointer=...)` plus a `thread_id` makes invokes one conversation.**

- after every step, the state is written to the saver
- `thread_id` in `config` picks which conversation to load
- `InMemorySaver` is the 1.x name; `MemorySaver` is the same class, an alias
- RAM only: gone on restart; the fix for that closes the session

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver  # MemorySaver: same class, old alias

threaded = support_builder.compile(checkpointer=InMemorySaver())
tcfg = {"configurable": {"thread_id": "ticket-1"}, "recursion_limit": 5}

one = threaded.invoke(
    {"messages": [HumanMessage("Hi. My order ORD-7431 arrived with a cracked screen.")]},
    config=tcfg,
)
two = threaded.invoke(
    {"messages": [HumanMessage("What was my order number again?")]},
    config=tcfg,  # same thread_id: turn 1 is loaded back in first
)
print("turn 1:", len(one["messages"]), "messages | turn 2:", len(two["messages"]))
print(two["messages"][0].content)  # straight from the checkpoint

**The silent failure: `thread_id` without a checkpointer.**

- the config is accepted; nothing warns, nothing is stored
- it looks exactly like "the model keeps forgetting"
- history vanishing between invokes? check `compile()` first

In [ ]:
forgetful = support_builder.compile()  # checkpointer missing, nothing else changed
fcfg = {"configurable": {"thread_id": "ticket-1"}, "recursion_limit": 5}

a = forgetful.invoke(
    {"messages": [HumanMessage("My order ORD-7431 arrived damaged.")]}, config=fcfg
)
b = forgetful.invoke(
    {"messages": [HumanMessage("What was my order number again?")]}, config=fcfg
)
print("turn 1:", len(a["messages"]), "| turn 2:", len(b["messages"]))  # no memory, no warning

In [ ]:
try:
    threaded.invoke(
        {"messages": [HumanMessage("hello")]},
        config={"recursion_limit": 5},  # checkpointer present, thread_id forgotten
    )
except ValueError as error:
    print("ValueError:", error)  # the inverse mistake at least fails loudly

**Send only the new message. The thread owns the history.**

- resending the whole client-side list duplicates it, silently
- `add_messages` dedupes by id, and dict messages get fresh ids
- the session-1 habit of rebuilding the list must die today

In [ ]:
dup_bot = support_builder.compile(checkpointer=InMemorySaver())
dcfg = {"configurable": {"thread_id": "ticket-dup"}, "recursion_limit": 5}

turn1 = {"role": "user", "content": "The tracking link for ORD-7431 is dead."}
state = dup_bot.invoke({"messages": [turn1]}, config=dcfg)
print("after turn 1:", len(state["messages"]), "messages")

turn2 = {"role": "user", "content": "Is the courier aware of it?"}
resent = dup_bot.invoke({"messages": [turn1, turn2]}, config=dcfg)  # the session-1 habit
print("after turn 2:", len(resent["messages"]), "messages, not 4")
print([m.content[:26] for m in resent["messages"]])  # turn 1 is in there twice

In [ ]:
snapshot = threaded.get_state(tcfg)  # the checkpoint itself, straight from the saver
print("stored for ticket-1:", len(snapshot.values["messages"]), "messages")

other = {"configurable": {"thread_id": "ticket-2"}, "recursion_limit": 5}
fresh = threaded.invoke(
    {"messages": [HumanMessage("New here. Where do returns go?")]}, config=other
)
print("ticket-2 sees:", len(fresh["messages"]), "messages")  # threads never share state

## State beyond messages

**Subclass `MessagesState`; plain fields replace on write.**

- a summary wants replacement, not appending: the default reducer is right
- reducers themselves are session-2 material; nothing new here
- the schema is where a memory policy gets room to live

In [ ]:
class TicketState(MessagesState):
    summary: str  # no reducer annotation: every write replaces


def note_summary(state: TicketState) -> dict:
    return {"summary": f"messages so far: {len(state['messages'])}"}


scratch = StateGraph(TicketState)
scratch.add_node("note", note_summary)
scratch.add_edge(START, "note")
scratch.add_edge("note", END)
noted = scratch.compile(checkpointer=InMemorySaver())

ncfg = {"configurable": {"thread_id": "notes"}, "recursion_limit": 5}
print(noted.invoke({"messages": [HumanMessage("first")]}, config=ncfg)["summary"])
print(noted.invoke({"messages": [HumanMessage("second")]}, config=ncfg)["summary"])

## The attention budget

**A bigger context is not more memory. Recall degrades as context grows.**

- Anthropic calls it context rot: attention is a finite budget, every token spends it
- the failure is quiet: vaguer answers, missed early facts
- so: measure first, then choose what to keep
- https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents

## Full buffer

**Keep everything. The checkpointed graph already does this.**

- lossless, zero machinery; the spend is linear
- `count_tokens_approximately`: pure Python, offline, good enough to steer by
- twelve scripted turns here stand in for the fifty in practice

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

DIALOGUE = [
    "Hi. My order ORD-7431 arrived with a cracked screen.",
    "It was a birthday gift, so this is quite urgent.",
    "I already tried a hard reset, nothing changed.",
    "The box itself looked fine, no dents anywhere.",
    "Yes, I kept all the original packaging.",
    "A replacement works for me, no refund needed.",
    "Weekday deliveries after six work best.",
    "Will the replacement carry a fresh warranty?",
    "Also, the invoice email never reached me.",
    "Please resend it to the address on file.",
    "One more thing: the courier left no note at all.",
    "Before we close: which order number do you have for me?",
]

buffer_bot = support_builder.compile(checkpointer=InMemorySaver())
bcfg = {"configurable": {"thread_id": "ticket-buffer"}, "recursion_limit": 5}

buffer_curve = []  # tokens per turn; the summarized run is judged against this
for turn, line in enumerate(DIALOGUE, 1):
    state = buffer_bot.invoke({"messages": [HumanMessage(line)]}, config=bcfg)
    tokens = count_tokens_approximately([SystemMessage(RULES)] + state["messages"])
    buffer_curve.append(tokens)
    print(f"turn {turn:2}: {len(state['messages']):2} messages, ~{tokens:3} tokens")

**Costs of the full buffer: none in machinery, all in tokens.**

- nothing is lost and nothing is managed
- the curve is linear; the bill and the attention budget follow it
- the right default, right up until it quietly is not

## Sliding window

**Keep the last N tokens. The front falls off.**

- `trim_messages` shapes the prompt at call time; state is untouched
- `include_system=True` pins the standing rules
- `start_on="human"` so the window never opens mid-exchange
- constant cost; the question is what fell off

In [ ]:
from langchain_core.messages import trim_messages


def support_window(messages: list) -> list:
    # the prompt the model will actually see: rules pinned, tail kept
    return trim_messages(
        [SystemMessage(RULES)] + messages,
        strategy="last",  # keep the end, drop the start
        max_tokens=120,
        token_counter=count_tokens_approximately,
        include_system=True,
        start_on="human",
    )


def call_support_windowed(state: MessagesState) -> dict:
    return {"messages": [model.invoke(support_window(state["messages"]))]}


window_builder = StateGraph(MessagesState)
window_builder.add_node("support", call_support_windowed)
window_builder.add_edge(START, "support")
window_builder.add_edge("support", END)
windowed_bot = window_builder.compile(checkpointer=InMemorySaver())

wcfg = {"configurable": {"thread_id": "ticket-window"}, "recursion_limit": 5}
for line in DIALOGUE:
    wstate = windowed_bot.invoke({"messages": [HumanMessage(line)]}, config=wcfg)

window = support_window(wstate["messages"])
print("checkpoint:", len(wstate["messages"]), "messages | window:", len(window))
print("ORD-7431 in checkpoint:", any("ORD-7431" in m.content for m in wstate["messages"]))
print("ORD-7431 in window:    ", any("ORD-7431" in m.content for m in window))

**The lost-instructions failure, shown structurally.**

- turn 1 held the order number; the window no longer does
- live, the agent politely re-asks for what it was already told
- `include_system` pins the rules, not early facts
- and the checkpoint still holds all 24 messages: the trim shaped the prompt only

## Editing the checkpoint

**Trim at call vs prune in state.**

- `trim_messages`: the checkpoint stays whole; you keep paying to store, not to send
- `RemoveMessage` through `add_messages`: history shrinks for good
- `REMOVE_ALL_MESSAGES` clears the whole list: the reset button

In [ ]:
from langchain_core.messages import AIMessage, RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES, add_messages

history = add_messages(
    [],
    [
        HumanMessage("Where is my refund?", id="m1"),
        AIMessage("Checking now.", id="m2"),
        HumanMessage("Thanks.", id="m3"),
    ],
)
pruned = add_messages(history, [RemoveMessage(id="m1")])  # a deletion is an update too
print("after RemoveMessage:      ", [m.id for m in pruned])

wiped = add_messages(history, [RemoveMessage(id=REMOVE_ALL_MESSAGES)])
print("after REMOVE_ALL_MESSAGES:", wiped)

## Summarization

**Replace the head with a summary. Keep the tail verbatim.**

- one summarize node, fired by message count: deterministic in class
- the summary rides in as a system message on every later call
- keeping the last exchange verbatim beside it is the buffer variant
- costs: sub-linear spend, but lossy, plus one model call per refresh

In [ ]:
def ticket_prompt(state: TicketState) -> list:
    prompt = [SystemMessage(RULES)]
    if state.get("summary"):
        prompt.append(SystemMessage(f"Summary so far: {state['summary']}"))
    return prompt + state["messages"]


def call_ticket_model(state: TicketState) -> dict:
    return {"messages": [model.invoke(ticket_prompt(state))]}


def summarize(state: TicketState) -> dict:
    ask = HumanMessage(
        "Fold the dialogue above into a short summary: facts, decisions, open questions."
    )
    prior = []  # feed the old summary back in, or its facts vanish
    if state.get("summary"):
        prior = [SystemMessage(f"Earlier summary: {state['summary']}")]
    digest = model.invoke(prior + state["messages"] + [ask])
    prune = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]  # keep last exchange
    return {"summary": digest.content, "messages": prune}


def needs_summary(state: TicketState) -> str:
    return "summarize" if len(state["messages"]) > 8 else END  # count, not tokens


ticket_builder = StateGraph(TicketState)
ticket_builder.add_node("model", call_ticket_model)
ticket_builder.add_node("summarize", summarize)
ticket_builder.add_edge(START, "model")
ticket_builder.add_conditional_edges("model", needs_summary, ["summarize", END])
ticket_builder.add_edge("summarize", END)
print(ticket_builder.compile().get_graph().draw_mermaid())

In [ ]:
summary_bot = ticket_builder.compile(checkpointer=InMemorySaver())
scfg = {"configurable": {"thread_id": "ticket-sum"}, "recursion_limit": 5}

print("turn   buffer   summarized   kept")
for turn, line in enumerate(DIALOGUE, 1):
    sstate = summary_bot.invoke({"messages": [HumanMessage(line)]}, config=scfg)
    tokens = count_tokens_approximately(ticket_prompt(sstate))  # the next call's prompt
    kept = len(sstate["messages"])
    print(f"{turn:4}   ~{buffer_curve[turn - 1]:4}   ~{tokens:8}   {kept:2} messages")

final = summary_bot.get_state(scfg).values
print("\nsummary in state:", bool(final.get("summary")))  # filled by two refreshes

**The summary prompt is load-bearing.**

- name what to keep: stated facts, decisions, open questions
- summarization can drop exactly the fact you will need later
- the saving lands next turn; the firing turn pays the full prompt plus the summary call
- the trigger here is message count; production usually counts tokens
- the prebuilt `SummarizationMiddleware`, named in session 3, automates exactly this move

## Hard token budget

**A strict cap, for when nothing else saved you.**

- same `trim_messages`, but the budget is a promise, never exceeded
- `allow_partial=True` would rather ship a fragment than blow the cap
- a last resort: pair it with a window or a summary, not instead of them

In [ ]:
full_history = buffer_bot.get_state(bcfg).values["messages"]
capped = trim_messages(
    [SystemMessage(RULES)] + full_history,
    strategy="last",
    max_tokens=80,  # a hard ceiling, not a comfort target
    token_counter=count_tokens_approximately,
    include_system=True,
    allow_partial=True,
    start_on="human",
)
print(len(full_history) + 1, "messages offered,", len(capped), "kept")
print("tokens sent:", count_tokens_approximately(capped), "and never above 80")

## Priority working memory

**Score what matters, pin it, evict the rest.**

- salience-scored entries survive; low scores age out
- costs: scoring latency every turn, and similarity is not importance
- `include_system=True` was pinning's simplest form, already on screen
- concept today; code and more techniques: https://github.com/NirDiamant/Agent_Memory_Techniques

## Choosing a technique

**The costs, one line per technique.**

- full buffer: lossless, zero machinery; linear spend per turn
- sliding window: constant spend; early facts silently fall off
- summarization: sub-linear; lossy, plus one model call per refresh

**Pick by what you can afford to lose.**

- summary plus verbatim tail: the gist and the last exchange; one more knob to tune
- hard budget: never over the cap; may cut mid-message; last resort
- priority memory: keeps what scores high; latency, and scoring can be wrong
- short dialogues: buffer. long factual ones: summary plus tail, cap on top

## SqliteSaver

**The same interface, a file instead of RAM.**

- separate package `langgraph-checkpoint-sqlite`, already in the course env
- the swap is one argument at compile time
- a fresh process reopens the file and the thread continues
- name your own `.db` file, and gitignore `*.db`

In [ ]:
import sqlite3
from pathlib import Path

from langgraph.checkpoint.sqlite import SqliteSaver

Path("tickets.db").unlink(missing_ok=True)  # fresh file, so reruns start clean

conn = sqlite3.connect("tickets.db", check_same_thread=False)
durable_bot = ticket_builder.compile(checkpointer=SqliteSaver(conn))  # the whole swap
dbcfg = {"configurable": {"thread_id": "ticket-db"}, "recursion_limit": 5}
for line in DIALOGUE[:5]:  # five turns: enough to make the summary fire once
    durable_bot.invoke({"messages": [HumanMessage(line)]}, config=dbcfg)

conn.close()  # the "restart": connection, saver and graph object all gone
del durable_bot

conn = sqlite3.connect("tickets.db", check_same_thread=False)
# from_conn_string is a context manager; direct construction suits notebooks
revived_bot = ticket_builder.compile(checkpointer=SqliteSaver(conn))
state = revived_bot.invoke({"messages": [HumanMessage(DIALOGUE[5])]}, config=dbcfg)
print("after the restart:", len(state["messages"]), "messages")  # read back off disk
print("summary survived: ", bool(state.get("summary")))

## Practice

**Your own assistant, in your own repository.**

1. compile with `InMemorySaver`; pass a `thread_id` on every invoke
2. send only the new message: delete any code that resends history
3. wire in summarization or a sliding window; be ready to defend the choice
4. swap to `SqliteSaver`, restart the Python process, prove the thread survived

**Then load-test it and read your own traces.**

5. run a 50-turn dialogue against your assistant, live
6. watch context size per turn in the traces
7. compare token spend, full buffer vs your technique, from your own traces
8. stretch: extract turn-1 facts into a pinned system line before trimming

**Required artifact: `runs/session-04.md`, committed.**

- token numbers before and after your technique, plus the restart transcript
- one exported trace in which the summarization node fires
- add `*.db` to your `.gitignore` before the first commit

In [ ]:
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # a fresh handler per run gives one trace per run
live = summary_bot.invoke(
    {"messages": [HumanMessage("Quick check: where is my replacement now?")]},
    config={
        "configurable": {"thread_id": "ticket-sum"},
        "recursion_limit": 5,
        "callbacks": [handler],
    },
)
client.flush()  # a notebook kernel never exits, so nothing sends without this

print(live["messages"][-1].content)  # if summarize fired, it is a span in the trace

**Individual project directions are discussed today, before you leave.**

- proposal, one page, due next session: the theme, and who the user is
- an architecture section, including today's context strategy
- the candidate quality metric; topics are approved and review pairs assigned next session

## Today, in one card

**Memory between invokes is a checkpointer plus a `thread_id`; what stays in the window is a policy you choose.**

**You can now defend:**
- `thread_id` without a checkpointer is accepted silently: nothing is stored, and it looks exactly like a forgetful model
- send only the new message: resending the client-side history duplicates it, because dict messages get fresh ids
- a window costs the same every turn and loses early facts; a summary is sub-linear, lossy, and one model call per refresh

**In your repository:** `runs/session-04.md`, token counts before and after your technique, the restart transcript, one trace where summarization fires.
**The trap of the day:** `trim_messages` shapes the prompt only; the checkpoint still holds every message, and you keep paying to store them.
**Ask yourself:** compare a sliding window, summarization and a hard token budget: what does each lose, and which agent wants which?
**Next time:** memory between dialogues: the Store, user profiles, trustcall, and the tools come back.